# 01 · Prepare ABO (reals + metadata + splits)

**Accelerator:** none · **Runtime:** ~30–45 min (3 GB download dominates) · **Output dataset name:** `abo-prepared`

Download ABO listings + small images (CC BY 4.0), clean the metadata (12-colour vocabulary, 8 materials, strict
"Set of N" counts, placeholder descriptions dropped), assign **family-based** splits, select the working subset
(7000 / 1000 / 2000 listings) and copy only those images into `data/processed/images`.

**Student intuition:** we never split by listing. The same sofa in four colours shares a *family key*, and the key
decides the split, so no template can appear on both sides. `split_report.json` reports brand overlap (reported,
not forbidden: the risk document's brand-disjoint wording is amended in the report).

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/kvamsi-iisc/DLVA-Final-Project-Forensic-Verification-of-Marketplace-Listings.git"   # used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if REPO.exists() and (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "-q"], check=False)     # pick up code fixes in a reused session
elif not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
print("python", sys.version.split()[0], "| code:", subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stdout[-3000:], pip.stderr[-3000:])          # show WHY before stopping (e.g. a pin without a wheel for this Python)
    raise SystemExit("pip install failed; see output above")
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)
sys.path.insert(0, str(REPO))
from listing_verifier.kaggle import init, run, attach, optional_attach, merge_fakes   # helpers live in the repo: fixes need no re-import
WORK, REPO, DATA, OUT, LOGS = init(WORK)
run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")
print("attached inputs:", sorted(p.name for p in pathlib.Path("/kaggle/input").iterdir()) if pathlib.Path("/kaggle/input").exists() else [])


In [ ]:
run("scripts/01_download_abo.py", "--images", "--raw-dir", DATA / "raw")

In [ ]:
run("scripts/02_prepare_abo.py", "--train", 7000, "--val", 1000, "--test", 2000, "--cap-per-type", 250, "--copy-images")

### Sanity-check the cleaning. If `n_with_colour` is below ~60% of `n_kept`, inspect `colour_raw` values and extend `COLOUR_TOKEN_MAP`. Quote the ACHIEVED counts in the report, never the requested ones.

In [ ]:
import pandas as pd
rep = json.load(open(DATA / "processed" / "cleaning_report.json")); print(json.dumps(rep, indent=1)[:3000])
print(json.load(open(DATA / "processed" / "split_report.json")))
df = pd.read_csv(DATA / "processed" / "abo_clean.csv")
print(df["split"].value_counts()); print(df["colour"].value_counts().head(12)); print(df["product_type"].value_counts().head(15))

### Look at 12 random listings with their labels (always look at your data).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
s = df.sample(12, random_state=1)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (_, r) in zip(axes.ravel(), s.iterrows()):
    ax.imshow(Image.open(DATA / "processed" / "images" / r["image_rel_path"])); ax.axis("off")
    ax.set_title(f"{r['colour']} | {r['material']} | n={r['count']}\n{str(r['item_name'])[:30]}", fontsize=8)
plt.tight_layout()

### Remove the raw 398k-image tree so the output dataset is just the working subset (~400 MB).

In [ ]:
shutil.rmtree(DATA / "raw", ignore_errors=True)

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["data/processed/abo_clean.csv", "data/processed/images", "data/processed/split_report.json"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)